In [52]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
while project_root.name != 'olist-delivery-risk':
    if project_root == project_root.parent:
        raise FileNotFoundError("找不到项目根目录 olist-delivery-risk")  
    project_root = project_root.parent

delay_orders = pd.read_csv(project_root/'data'/'processed'/'orders_with_delay.csv')
date_cols = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date']
delay_orders[date_cols] = delay_orders[date_cols].apply(pd.to_datetime, errors="coerce")

print(delay_orders.dtypes)

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
item_count                              float64
product_amount                          float64
freight_amount                          float64
total_amount                            float64
customer_unique_id                          str
customer_zip_code_prefix                  int64
customer_city                               str
customer_state                              str
month                                       str
is_delayed                                int64
dtype: object


In [53]:
# 定义候选特征
candidate_features = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_estimated_delivery_date",
    "customer_state",
    "customer_city",
    "customer_zip_code_prefix",
    "item_count",
    "product_amount",
    "freight_amount",
    "total_amount"
]

# 统计每个候选字段的缺失情况
missing_summary = pd.DataFrame({
     "missing_count":delay_orders[candidate_features].isna().sum(),
     "missing_rate":delay_orders[candidate_features].isna().mean()*100
})
missing_summary 

,missing_count,missing_rate
order_purchase_timestamp,0,0.0
order_approved_at,0,0.0
order_estimated_delivery_date,0,0.0
customer_state,0,0.0
customer_city,0,0.0
customer_zip_code_prefix,0,0.0
item_count,0,0.0
product_amount,0,0.0
freight_amount,0,0.0
total_amount,0,0.0


In [54]:
# 定义禁止作为模型输入的字段
forbidden_features = [
    'order_id',                      # 订单唯一标识
    'customer_id',                   # 客户订单标识
    'customer_unique_id',            # 客户身份标识
    'order_status',                  # 最终订单状态
    'order_delivered_carrier_date',  # 实际交付承运商时间
    'order_delivered_customer_date', # 实际送达时间
    'is_delayed'                     # 预测标签，不能作为输入特征
]

# 检查候选特征和禁用字段是否存在交集
leakage_check = set(candidate_features) & set(forbidden_features)

print('候选特征中的禁用字段:', leakage_check)

候选特征中的禁用字段: set()


In [55]:
valid = (
    delay_orders["order_status"].eq("delivered")
    & delay_orders["order_approved_at"].notna()
    & delay_orders["order_delivered_customer_date"].notna()
    & delay_orders["order_estimated_delivery_date"].notna()
)

expected_label = (
    delay_orders["order_delivered_customer_date"].dt.normalize()
    > delay_orders["order_estimated_delivery_date"].dt.normalize()
).astype(int)

print("不符合样本规则的数量：", (~valid).sum())

print(
    "延迟标签不一致的数量：",
    (delay_orders["is_delayed"] != expected_label).sum()
)

不符合样本规则的数量： 0
延迟标签不一致的数量： 0


In [56]:
# 复制一份建模数据
ml_data = delay_orders.copy()

# 下单时间特征
ml_data["purchase_month"] = (
    ml_data["order_purchase_timestamp"].dt.month
)

ml_data["purchase_weekday"] = (
    ml_data["order_purchase_timestamp"].dt.dayofweek
)

ml_data["purchase_hour"] = (
    ml_data["order_purchase_timestamp"].dt.hour
)

# 订单获批时间特征
ml_data["approval_hour"] = (
    ml_data["order_approved_at"].dt.hour
)

In [57]:
model_features = [
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "approval_hour",
    "customer_state",
    "customer_city",
    "item_count",
    "product_amount",
    "freight_amount"
]

# 确认选择的特征没有包含禁用字段
assert not (set(model_features) & set(forbidden_features))

# 构建特征矩阵 X
X = ml_data[model_features].copy()

# 构建目标标签 y
y = ml_data["is_delayed"].copy()

print("X 的形状：", X.shape)
print("y 的形状：", y.shape)

X 的形状： (96456, 9)
y 的形状： (96456,)


In [58]:
# 检查特征缺失情况
model_missing = pd.DataFrame({
    "missing_count": X.isna().sum(),
    "missing_rate": X.isna().mean() * 100
})

print(model_missing)

# 检查标签是否有缺失
print("标签缺失数：", y.isna().sum())

# 检查标签分布
print(y.value_counts())

# 检查延迟比例
print(y.value_counts(normalize=True))

# 检查样本数量是否一致
print("X和y行数一致：", len(X) == len(y))

# 检查城市类别数量
print("城市类别数量：", X["customer_city"].nunique())

                  missing_count  missing_rate
purchase_month                0           0.0
purchase_weekday              0           0.0
purchase_hour                 0           0.0
approval_hour                 0           0.0
customer_state                0           0.0
customer_city                 0           0.0
item_count                    0           0.0
product_amount                0           0.0
freight_amount                0           0.0
标签缺失数： 0
is_delayed
0    89922
1     6534
Name: count, dtype: int64
is_delayed
0    0.932259
1    0.067741
Name: proportion, dtype: float64
X和y行数一致： True
城市类别数量： 4085


In [59]:
model_dataset = ml_data[
     ["order_id", "order_approved_at"]
     + model_features
     + ["is_delayed"]
]

#路径
output_path = (project_root/'data'/'processed'/'model_dataset.csv')

#保存
model_dataset.to_csv(output_path,index=False)
print("保存成功：", output_path)
print("数据形状：", model_dataset.shape)

保存成功： d:\各类文件\项目\olist-delivery-risk\data\processed\model_dataset.csv
数据形状： (96456, 12)


### 1. 预测任务定义

本阶段基于已构建的订单级数据集 `orders_with_delay`，开展机器学习建模前的数据准备工作。

**预测目标：**

在订单获批时，根据已知的订单、商品及客户信息，预测订单最终是否发生配送延迟。

本项目属于二分类问题：

- `is_delayed = 0`：订单未延迟。
- `is_delayed = 1`：订单发生延迟。

延迟标签采用日历日期比较规则：实际送达日期晚于预计送达日期，则判定为延迟。

## 2. 候选特征筛选

首先从订单级数据集中选取可能与配送延迟相关的原始字段，包括订单时间、客户地区、商品数量及金额信息。

对原始候选字段进行缺失值检查，并根据预测时点判断各字段是否可以用于模型。

为防止数据泄漏，排除以下字段：

- `order_id`
- `customer_id`
- `customer_unique_id`
- `order_status`
- `order_delivered_carrier_date`
- `order_delivered_customer_date`
- `is_delayed`

其中，实际送达时间和交付承运商时间属于预测时点之后的信息，不能作为模型输入；`is_delayed` 为预测目标变量。

同时，暂不使用 `order_estimated_delivery_date`，因为仍需确认其在订单获批时是否已知。

## 3. 时间特征工程

从原始时间字段中提取以下特征：

| 特征 | 含义 |
|---|---|
| `purchase_month` | 下单月份 |
| `purchase_weekday` | 下单星期 |
| `purchase_hour` | 下单小时 |
| `approval_hour` | 订单获批小时 |

通过时间特征提取，将原始时间戳转换为模型可以使用的数值特征。

## 4. 第一版模型特征

最终选择 9 个特征，构成模型输入矩阵 `X`。

| 特征类别 | 字段 | 数量 |
|---|---|---:|
| 时间信息 | `purchase_month`、`purchase_weekday`、`purchase_hour`、`approval_hour` | 4 |
| 地区信息 | `customer_state`、`customer_city` | 2 |
| 商品信息 | `item_count`、`product_amount`、`freight_amount` | 3 |
| **合计** | | **9** |

其中，商品数量、商品金额和运费的使用基于一个业务假设：这些信息在订单获批时已经确定。

城市与州同时保留，以利用不同层级的地区信息。后续可通过验证集比较加入城市特征前后的模型表现。

## 5. 建模数据检查

### 5.1 数据规模

根据本次检查结果：

| 项目 | 结果 |
|---|---:|
| 建模样本数量 | 96,456 |
| 输入特征数量 | 9 |
| 目标标签数量 | 96,456 |
| `X` 和 `y` 行数是否一致 | True |

说明特征矩阵与标签数量一致。

### 5.2 特征缺失值检查

对最终选取的 9 个模型特征进行缺失值统计：

| 特征 | 缺失数量 | 缺失比例 |
|---|---:|---:|
| `purchase_month` | 0 | 0.00% |
| `purchase_weekday` | 0 | 0.00% |
| `purchase_hour` | 0 | 0.00% |
| `approval_hour` | 0 | 0.00% |
| `customer_state` | 0 | 0.00% |
| `customer_city` | 0 | 0.00% |
| `item_count` | 0 | 0.00% |
| `product_amount` | 0 | 0.00% |
| `freight_amount` | 0 | 0.00% |

**检查结论：**

本次建模数据的 9 个候选输入特征均不存在缺失值。

后续可以直接开展数据划分，但仍需要根据模型类型进行类别编码等预处理。

这里的完整性仅指本次选取的模型特征，不代表原始数据不存在其他缺失或异常。

### 5.3 目标标签分布

统计 `is_delayed` 的数量和比例：

| 标签 | 含义 | 订单数量 | 占比 |
|---|---|---:|---:|
| 0 | 未延迟 | 89,922 | 93.23% |
| 1 | 延迟 | 6,534 | 6.77% |
| **合计** | | **96,456** | **100.00%** |

整体延迟率：

\[
\text{Delay Rate}=\frac{6534}{96456}\times100\%\approx6.77\%
\]

**分析结论：**

1. 未延迟订单占绝大多数，比例约为 93.23%。
2. 延迟订单占比约为 6.77%，属于明显的类别不平衡问题。
3. 后续模型评估不能只依赖准确率（Accuracy），还应关注 Average Precision（AP）、精确率（Precision）和召回率（Recall）。
4. 模型需要与简单基线进行比较，判断其是否能够有效识别少数类延迟订单。

### 5.4 城市类别检查

统计结果：

| 项目 | 结果 |
|---|---:|
| `customer_city` 不同取值数量 | 4,085 |

**分析结论：**

客户城市属于高基数类别特征。

如果直接使用 One-Hot 编码，可能产生大量编码列，并且部分城市的订单样本较少。

因此，后续建模时需要考虑：

- 使用 `OneHotEncoder` 进行类别编码；
- 对未见过的城市类别进行合理处理；
- 视训练集类别频数决定是否合并低频城市；
- 比较仅使用州和同时使用州、城市两种特征方案的表现。

类别编码器应仅在训练集上拟合，防止验证集和测试集信息泄漏。

## 6. 数据泄漏控制

本阶段已通过禁用字段清单对模型候选字段进行初步检查。

同时明确：

- 不使用实际送达时间等未来信息；
- 不将目标标签作为输入；
- 不直接使用最终订单状态；
- 不将订单唯一标识作为模型特征；
- 商品金额等字段的预测时点可用性仍依赖业务假设。

禁用字段交集检查只能发现清单内的违规字段，后续新增特征仍需单独判断其信息来源。

## 7. 当前数据局限

1. 数据中仍可能存在此前检查发现但尚未处理的异常值。
2. 商品相关特征在订单获批时的可用性需要依赖业务假设。
3. 城市类别数量较多，存在类别稀疏和过拟合风险。
4. 延迟订单占比较低，模型评估需要考虑类别不平衡。
5. 当前统计结果尚不能单独证明所有样本均满足预测任务的筛选条件，仍需结合上游数据生成代码核对。

## 8. 本阶段总结

第 8 天完成了订单延迟预测任务的初步定义与特征工程。

主要成果：

1. 明确了模型预测目标与预测时点。
2. 整理了候选特征和禁用字段。
3. 提取了 4 个时间特征。
4. 确定了包含时间、地区和商品信息的 9 个模型特征。
5. 构建了特征矩阵 `X` 和目标标签 `y`。
6. 验证了两者样本数量一致。
7. 确认 9 个模型特征均无缺失值。
8. 发现延迟订单比例约为 6.77%，存在类别不平衡。
9. 发现城市字段包含 4,085 个类别，后续需合理编码。

## 9. 下一阶段工作

第 9 天将基于订单获批时间 `order_approved_at`，按照时间顺序划分：

- 训练集（Training Set）
- 验证集（Validation Set）
- 测试集（Test Set）

重点工作包括：

1. 检查订单获批时间分布。
2. 确定训练、验证和测试集时间边界。
3. 检查各数据集的样本数量和延迟比例。
4. 检查历史样本标签在相应时间点是否已经可知。
5. 为后续基线模型和逻辑回归模型训练做好准备。